# 🏆 Simulation Coupe du Monde 2026
## Système de prédiction basé sur des données réelles

> **Objectif :** Simuler l'intégralité de la CdM 2026 — phase de groupes, Round of 32, quarts, demis, finale — à partir d'un modèle de score composite et d'une simulation Monte Carlo (10 000 itérations).
|

### Format CdM 2026
- 48 équipes · 12 groupes de 4 · Top 2 de chaque groupe + 8 meilleurs 3ᵉ = **32 équipes**
- Round of 32 → 1/4 de finale → 1/2 finale → Finale (MetLife Stadium, 19 juillet 2026)

---

## 1. Importation des bibliothèques

Import des bibliothèques principales utilisées dans le notebook :

- `pandas` pour la manipulation de données
- `numpy` pour les calculs numériques

(Exécuter la cellule pour vérifier les versions si besoin.)

In [ ]:
import pandas as pd
import numpy as np

## 2. Chargement des données

Lecture des fichiers sources nécessaires à la simulation et affectation aux DataFrame :

- `valeurs_marchandes.csv` → `df_valeurs`
- `resultats_depuis_2021.csv` → `df_resultats`
- `meilleure_defense.csv` → `df_defenses`
- `classement_fifa.csv` → `df_classement`
- `buts_marques_depuis_2021.csv` → `df_buts`

Exécuter la cellule pour charger les données.

In [ ]:
df_valeurs = pd.read_csv('valeurs_marchandes.csv')
df_resultats = pd.read_csv('resultats_depuis_2021.csv')
df_defenses = pd.read_csv('meilleure_defense.csv')
df_classement= pd.read_csv('classement_fifa.csv')
df_buts = pd.read_csv('buts_marques_depuis_2021.csv')

## 3. Fusion et préparation des données

Construction du `DataFrame` maître `df_master` en fusionnant les sources sur la colonne `team` :

- `df_classement` × `df_valeurs` → `df_intermediaire`
- + `df_resultats` → `df_int`
- + `df_defenses` → `df_int_2`
- + `df_buts` → `df_master`

Affichage des premières lignes pour vérification.

In [ ]:
df_intermediaire=df_classement.merge(df_valeurs,on='team')
df_int=df_intermediaire.merge(df_resultats,on='team')
df_int_2=df_int.merge(df_defenses,on='team')
df_master=df_int_2.merge(df_buts,on='team')
df_master.head()

## 4. Calcul des indicateurs de performance

Création de nouvelles colonnes normalisées dans `df_master` :

- `taux_victoires` = `victoires` / `matchs_officiels_joues`
- `taux_invincibilite` = (`victoires` + `nuls`) / `matchs_officiels_joues`
- `buts_encaisses_par_match` = `buts_encaisses_depuis_2021` / `matchs_officiels_joues`
- `buts_marques_par_match` = `buts_marques_depuis_2021` / `matchs_officiels_joues`

Afficher `df_master.head()` pour valider.

In [ ]:
df_master["taux_victoires"]=df_master["victoires"]/df_master["matchs_officiels_joues"]
df_master["taux_invincibilite"]=(df_master["victoires"]+df_master['nuls'])/df_master["matchs_officiels_joues"]
df_master["buts_encaisses_par_match"]=df_master["buts_encaisses_depuis_2021"]/df_master["matchs_officiels_joues"]
df_master["buts_marques_par_match"]=df_master["buts_marques_depuis_2021"]/df_master["matchs_officiels_joues"]
df_master.head()

## 5. Normalisation du classement FIFA

Création de `score_fifa_norme` : normalisation inversée de `classement_fifa` (un classement plus bas → meilleur score normalisé). On affiche ensuite `df_master.head()` pour contrôle.

In [ ]:
df_master['score_fifa_norme']=(max(df_master['classement_fifa'])-df_master['classement_fifa'])/(max(df_master['classement_fifa'])-min(df_master['classement_fifa']))
df_master.head()   

## 6. Transformation et normalisation de la valeur marchande

Calcul de `valeur_log` = log10(`valeur_marchande_euros`) puis normalisation en `valeur_log_norme`. Cette transformation réduit l'étalement des valeurs extrêmes.

In [ ]:
df_master['valeur_log']=np.log10(df_master['valeur_marchande_euros'])
df_master['valeur_log_norme']=(df_master['valeur_log']-min(df_master['valeur_log']))/(max(df_master['valeur_log'])-min(df_master['valeur_log']))
df_master.head()

## 7. Solidité défensive normalisée

Calcul de `solidite_defensive` : on inverse `buts_encaisses_par_match` puis on normalise pour obtenir un score où une valeur plus élevée signifie meilleure solidité.

In [ ]:
df_master['solidite_defensive']=(max(df_master['buts_encaisses_par_match'])-df_master['buts_encaisses_par_match'])/(max(df_master['buts_encaisses_par_match'])-min(df_master['buts_encaisses_par_match']))
df_master.head()

## 8. Normalisation des buts marqués

Normalisation de `buts_marques_par_match` en `buts_marques_par_match_norme` pour rendre les valeurs comparables entre équipes.

In [ ]:
df_master['buts_marques_par_match_norme']=(df_master['buts_marques_par_match']-min(df_master['buts_marques_par_match']))/(max(df_master['buts_marques_par_match'])-min(df_master['buts_marques_par_match']))
df_master.head()

## 9. Indices d'attaque et de défense

Calcul des indices synthétiques :

- `puissance_attaque` : combinaison pondérée des buts normés, taux de victoires, valeur marchande normalisée et score FIFA.
- `puissance_defense` : combinaison pondérée de la solidité défensive, taux d'invincibilité et score FIFA.

Afficher `df_master.head()` pour vérification.

In [ ]:
df_master['puissance_attaque']=df_master['buts_marques_par_match_norme']*0.4+df_master['taux_victoires']*0.3+df_master['valeur_log_norme']*0.2+df_master['score_fifa_norme']*0.1
df_master['puissance_defense']=df_master['solidite_defensive']*0.5+df_master['taux_invincibilite']*0.3+df_master['score_fifa_norme']*0.2
df_master.head()